# Used Car Price Prediction
## End-to-End Machine Learning Block

**Dataset:** Used Car Price Prediction Dataset (Kaggle – taeefnajib)

**Goal:** Predict the market price of a used car based on structured features (year, mileage, brand, fuel type, etc.) and a condition score derived from the Computer Vision block.

**Integration:** The `condition_score` output by the CV block (0 = unknown/undamaged, 1 = minor damage, 2 = major damage) is used as a feature here. The final predicted price is passed to the NLP block to generate a natural language explanation.

## Project Setup

### Libraries and Settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pickle
import warnings

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

## 1. Data Loading and Inspection

**Data source:** [Used Car Price Prediction Dataset](https://www.kaggle.com/datasets/taeefnajib/used-car-price-prediction-dataset)

Download `used_car_price_prediction.csv` from Kaggle and place it in the same folder as this notebook.
On Kaggle notebooks the path is: `/kaggle/input/used-car-price-prediction-dataset/used_car_price_prediction.csv`

In [ ]:
import os

# Support both local and Kaggle environments
kaggle_path = '/kaggle/input/datasets/taeefnajib/used-car-price-prediction-dataset/used_cars.csv'
local_path  = 'used_car_price_prediction.csv'

data_path = kaggle_path if os.path.exists(kaggle_path) else local_path

df_full = pd.read_csv(data_path)
print('Shape:', df_full.shape)
df_full.head(5)

In [ ]:
print('Column types:')
print(df_full.dtypes)
print('\nMissing values:')
print(df_full.isnull().sum())

In [ ]:
df_full.describe()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Price distribution (raw string column — parse first for plotting)
prices_raw = df_full['price'].astype(str).str.replace(r'[\$,]', '', regex=True)
prices_num = pd.to_numeric(prices_raw, errors='coerce').dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(prices_num, bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution (raw)')
axes[0].set_xlabel('Price (USD)')
axes[0].set_ylabel('Count')

axes[1].hist(prices_num[prices_num < 100000], bins=60, color='darkorange', edgecolor='white')
axes[1].set_title('Price Distribution (< $100k)')
axes[1].set_xlabel('Price (USD)')
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 brands by listing count
if 'brand' in df_full.columns:
    brand_counts = df_full['brand'].value_counts().head(15)
    brand_counts.plot(kind='barh', figsize=(8, 5), color='steelblue')
    plt.title('Top 15 Car Brands by Listing Count')
    plt.xlabel('Count')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# Fuel type and transmission distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if 'fuel_type' in df_full.columns:
    df_full['fuel_type'].value_counts().plot(kind='bar', ax=axes[0], color='teal', edgecolor='white')
    axes[0].set_title('Fuel Type Distribution')
    axes[0].set_xlabel('')

if 'transmission' in df_full.columns:
    df_full['transmission'].value_counts().plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
    axes[1].set_title('Transmission Distribution')
    axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

## 3. Data Cleaning and Preprocessing

In [ ]:
df = df_full.copy()

# --- Parse price: remove '$' and ',' → float ---
df['price'] = df['price'].astype(str).str.replace(r'[\$,]', '', regex=True)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# --- Parse milage: remove 'mi.' and ',' → float ---
if 'milage' in df.columns:
    df['milage'] = df['milage'].astype(str).str.replace(r'[mi\.\s,]', '', regex=True)
    df['milage'] = pd.to_numeric(df['milage'], errors='coerce')

# --- Parse engine HP: extract first numeric value ---
if 'engine' in df.columns:
    df['engine_hp'] = df['engine'].astype(str).str.extract(r'(\d+\.?\d*)\s*HP', flags=re.IGNORECASE)
    df['engine_hp'] = pd.to_numeric(df['engine_hp'], errors='coerce')

print('After parsing — shape:', df.shape)
df[['price', 'milage', 'engine_hp']].describe()

In [ ]:
# Remove missing values in critical columns
df = df.dropna(subset=['price', 'milage', 'model_year'])

# Remove duplicates
df = df.drop_duplicates()

# Remove price outliers (keep between $1,000 and $200,000)
df = df[(df['price'] >= 1000) & (df['price'] <= 200000)]

# Remove mileage outliers (keep below 500,000 mi)
df = df[df['milage'] <= 500000]

print('After cleaning — shape:', df.shape)
df.head(3)

## 4. Feature Engineering

Applied progressively across iterations.

In [ ]:
# --- Derived numeric features ---
current_year = 2024
df['car_age'] = current_year - df['model_year']                    # age of car in years
df['age_times_milage'] = df['car_age'] * df['milage']              # interaction: depreciation proxy

# --- Binary flags from categorical columns ---
if 'accident' in df.columns:
    df['has_accident'] = df['accident'].astype(str).str.contains('accident', case=False).astype(int)

if 'clean_title' in df.columns:
    df['clean_title_flag'] = (df['clean_title'].astype(str).str.strip().str.lower() == 'yes').astype(int)

# --- Label encode fuel_type and transmission ---
le_fuel = LabelEncoder()
le_trans = LabelEncoder()

if 'fuel_type' in df.columns:
    df['fuel_type_enc'] = le_fuel.fit_transform(df['fuel_type'].astype(str))
if 'transmission' in df.columns:
    df['transmission_enc'] = le_trans.fit_transform(df['transmission'].astype(str))

# --- Encode top 20 brands, group rest as 'Other' ---
if 'brand' in df.columns:
    top_brands = df['brand'].value_counts().head(20).index.tolist()
    df['brand_grouped'] = df['brand'].apply(lambda x: x if x in top_brands else 'Other')
    le_brand = LabelEncoder()
    df['brand_enc'] = le_brand.fit_transform(df['brand_grouped'].astype(str))

# --- condition_score placeholder (0 = not yet assessed by CV block) ---
# This column will be filled by the Computer Vision block at inference time.
# 0 = no damage / unknown, 1 = minor damage, 2 = major damage
df['condition_score'] = 0

print('Feature columns added:')
print([c for c in df.columns if c not in df_full.columns])

### Utility Functions

In [ ]:
def model_performance_lr(features, df, model=None):
    if model is None:
        model = LinearRegression()
    df = df.sample(frac=1, random_state=42)
    X, y = df[features], df['price']
    scores = cross_val_score(model, X, y, scoring='neg_root_mean_squared_error', cv=5)
    print('CV RMSE:', np.round(np.abs(scores)))
    print('Mean RMSE:', np.round(np.abs(scores).mean(), 1))

def model_performance_rf(features, df, model=None):
    if model is None:
        model = RandomForestRegressor(random_state=42)
    df = df.sample(frac=1, random_state=42)
    X, y = df[features], df['price']
    scores = cross_val_score(model, X, y, scoring='neg_root_mean_squared_error', cv=5)
    print('CV RMSE:', np.round(np.abs(scores)))
    print('Mean RMSE:', np.round(np.abs(scores).mean(), 1))

def plot_feature_importance(model, features):
    df_fi = pd.DataFrame({'feature': features, 'importance': model.feature_importances_})
    df_fi = df_fi.sort_values('importance')
    df_fi.plot(kind='barh', x='feature', y='importance', figsize=(7, 4),
               color='steelblue', legend=False)
    plt.title('Feature Importances')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

---
## Iteration 1 — Baseline: Core Numeric Features

**Objective:** Establish a baseline using only the most obvious numeric features (model year and mileage).

**Features:** `model_year`, `milage`

**Models:** Linear Regression, Random Forest Regressor

In [ ]:
features_iter1 = ['model_year', 'milage']

X = df[features_iter1]
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Linear Regression ---
lr_1 = LinearRegression()
lr_1.fit(X_train, y_train)
print('=== Linear Regression ===')
print('Train R²:', round(lr_1.score(X_train, y_train), 4))
print('Test  R²:', round(lr_1.score(X_test, y_test), 4))
print('Train RMSE:', round(root_mean_squared_error(y_train, lr_1.predict(X_train)), 1))
print('Test  RMSE:', round(root_mean_squared_error(y_test, lr_1.predict(X_test)), 1))
print()
model_performance_lr(features_iter1, df)

print()

# --- Random Forest ---
rf_1 = RandomForestRegressor(random_state=42)
rf_1.fit(X_train, y_train)
print('=== Random Forest ===')
print('Train R²:', round(rf_1.score(X_train, y_train), 4))
print('Test  R²:', round(rf_1.score(X_test, y_test), 4))
print('Train RMSE:', round(root_mean_squared_error(y_train, rf_1.predict(X_train)), 1))
print('Test  RMSE:', round(root_mean_squared_error(y_test, rf_1.predict(X_test)), 1))
print()
model_performance_rf(features_iter1, df)

In [ ]:
plot_feature_importance(rf_1, features_iter1)

---
## Iteration 2 — Extended Features: Categorical Encoding + Feature Engineering

**Objective:** Add brand, fuel type, transmission, accident history, and derived features to improve the model.

**Key changes:** Label-encoded categoricals, `car_age`, `age_times_milage`, `has_accident`, `clean_title_flag`

**Models:** Linear Regression, Random Forest Regressor

In [ ]:
features_iter2 = [
    'model_year', 'milage', 'car_age', 'age_times_milage',
    'fuel_type_enc', 'transmission_enc', 'brand_enc',
    'has_accident', 'clean_title_flag'
]
# Only keep features that exist (defensive against different dataset versions)
features_iter2 = [f for f in features_iter2 if f in df.columns]

df_iter2 = df.dropna(subset=features_iter2 + ['price'])
X = df_iter2[features_iter2]
y = df_iter2['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Linear Regression ---
lr_2 = LinearRegression()
lr_2.fit(X_train, y_train)
print('=== Linear Regression ===')
print('Train R²:', round(lr_2.score(X_train, y_train), 4))
print('Test  R²:', round(lr_2.score(X_test, y_test), 4))
print('Train RMSE:', round(root_mean_squared_error(y_train, lr_2.predict(X_train)), 1))
print('Test  RMSE:', round(root_mean_squared_error(y_test, lr_2.predict(X_test)), 1))
print()
model_performance_lr(features_iter2, df_iter2)

print()

# --- Random Forest ---
rf_2 = RandomForestRegressor(random_state=42)
rf_2.fit(X_train, y_train)
print('=== Random Forest ===')
print('Train R²:', round(rf_2.score(X_train, y_train), 4))
print('Test  R²:', round(rf_2.score(X_test, y_test), 4))
print('Train RMSE:', round(root_mean_squared_error(y_train, rf_2.predict(X_train)), 1))
print('Test  RMSE:', round(root_mean_squared_error(y_test, rf_2.predict(X_test)), 1))
print()
model_performance_rf(features_iter2, df_iter2)

In [ ]:
plot_feature_importance(rf_2, features_iter2)

---
## Iteration 3 — Full Feature Set with condition_score + Gradient Boosting

**Objective:** Add `engine_hp` and `condition_score` (from CV block). Introduce Gradient Boosting as a third model to compare against Random Forest.

**Key changes:** `engine_hp`, `condition_score` added; Gradient Boosting Regressor introduced

**Models:** Random Forest, Gradient Boosting Regressor

In [ ]:
features_iter3 = [
    'model_year', 'milage', 'car_age', 'age_times_milage',
    'fuel_type_enc', 'transmission_enc', 'brand_enc',
    'has_accident', 'clean_title_flag',
    'engine_hp', 'condition_score'
]
features_iter3 = [f for f in features_iter3 if f in df.columns]

df_iter3 = df.dropna(subset=features_iter3 + ['price'])
X = df_iter3[features_iter3]
y = df_iter3['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Random Forest (best from Iter 2, now with more features) ---
rf_3 = RandomForestRegressor(n_estimators=200, random_state=42)
rf_3.fit(X_train, y_train)
print('=== Random Forest (n=200) ===')
print('Train R²:', round(rf_3.score(X_train, y_train), 4))
print('Test  R²:', round(rf_3.score(X_test, y_test), 4))
print('Train RMSE:', round(root_mean_squared_error(y_train, rf_3.predict(X_train)), 1))
print('Test  RMSE:', round(root_mean_squared_error(y_test, rf_3.predict(X_test)), 1))
print()
model_performance_rf(features_iter3, df_iter3, rf_3)

print()

# --- Gradient Boosting ---
gb_3 = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42)
gb_3.fit(X_train, y_train)
print('=== Gradient Boosting ===')
print('Train R²:', round(gb_3.score(X_train, y_train), 4))
print('Test  R²:', round(gb_3.score(X_test, y_test), 4))
print('Train RMSE:', round(root_mean_squared_error(y_train, gb_3.predict(X_train)), 1))
print('Test  RMSE:', round(root_mean_squared_error(y_test, gb_3.predict(X_test)), 1))
print()
model_performance_rf(features_iter3, df_iter3, gb_3)

In [ ]:
plot_feature_importance(rf_3, features_iter3)

In [ ]:
# --- Summary table of all iterations ---
# Re-create test sets per iteration (same random_state=42 ensures identical splits)
_, X_t1, _, y_t1 = train_test_split(df[features_iter1], df['price'], test_size=0.2, random_state=42)
_, X_t2, _, y_t2 = train_test_split(df_iter2[features_iter2], df_iter2['price'], test_size=0.2, random_state=42)
_, X_t3, _, y_t3 = train_test_split(df_iter3[features_iter3], df_iter3['price'], test_size=0.2, random_state=42)

summary = pd.DataFrame({
    'Iteration': ['1 – LR baseline', '1 – RF baseline',
                  '2 – LR extended', '2 – RF extended',
                  '3 – RF full',     '3 – GB full'],
    'Test R²': [
        round(lr_1.score(X_t1, y_t1), 4),
        round(rf_1.score(X_t1, y_t1), 4),
        round(lr_2.score(X_t2, y_t2), 4),
        round(rf_2.score(X_t2, y_t2), 4),
        round(rf_3.score(X_t3, y_t3), 4),
        round(gb_3.score(X_t3, y_t3), 4),
    ],
    'Test RMSE (USD)': [
        round(root_mean_squared_error(y_t1, lr_1.predict(X_t1)), 0),
        round(root_mean_squared_error(y_t1, rf_1.predict(X_t1)), 0),
        round(root_mean_squared_error(y_t2, lr_2.predict(X_t2)), 0),
        round(root_mean_squared_error(y_t2, rf_2.predict(X_t2)), 0),
        round(root_mean_squared_error(y_t3, rf_3.predict(X_t3)), 0),
        round(root_mean_squared_error(y_t3, gb_3.predict(X_t3)), 0),
    ]
})
print(summary.to_string(index=False))

## 5. Error Analysis

In [ ]:
# Use best model (rf_3) for error analysis
y_pred = rf_3.predict(X_test)
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Residuals vs Actual
axes[0].scatter(y_test, residuals, alpha=0.4, color='steelblue', s=10)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Actual Price (USD)')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Actual Price')

# Residual histogram
axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Residual (USD)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Histogram of Residuals')

plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted scatter
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=10, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', label='Perfect prediction')
plt.xlabel('Actual Price (USD)')
plt.ylabel('Predicted Price (USD)')
plt.title('Actual vs Predicted (Random Forest, Iter 3)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Worst predictions
df_err = X_test.copy()
df_err['actual_price']    = y_test.values
df_err['predicted_price'] = y_pred
df_err['abs_error']       = np.abs(residuals.values)
print('Top 10 worst predictions:')
print(df_err.sort_values('abs_error', ascending=False).head(10)[['actual_price', 'predicted_price', 'abs_error']])

## 6. Save Final Model

In [ ]:
# Save the best model (Random Forest, Iteration 3) and the feature list
model_payload = {
    'model': rf_3,
    'features': features_iter3,
    'label_encoders': {
        'fuel_type': le_fuel if 'fuel_type' in df.columns else None,
        'transmission': le_trans if 'transmission' in df.columns else None,
        'brand': le_brand if 'brand' in df.columns else None,
    }
}

with open('car_price_model.pkl', 'wb') as f:
    pickle.dump(model_payload, f)

print('Model saved as car_price_model.pkl')
print('Features used:', features_iter3)

## 7. Integration Test — Predict a Single Car

Demonstrates how the CV block's `condition_score` plugs into this model at inference time.

In [ ]:
def predict_car_price(model_payload, model_year, milage, fuel_type, transmission,
                      brand, has_accident, clean_title, engine_hp, condition_score):
    """
    Predict price for a single car.
    condition_score: 0 = no damage, 1 = minor, 2 = major  (output from CV block)
    """
    le = model_payload['label_encoders']
    model = model_payload['model']
    features = model_payload['features']

    car_age = 2024 - model_year
    age_times_milage = car_age * milage

    fuel_enc  = le['fuel_type'].transform([fuel_type])[0]   if le['fuel_type']  else 0
    trans_enc = le['transmission'].transform([transmission])[0] if le['transmission'] else 0
    brand_enc = le['brand'].transform([brand if brand in le['brand'].classes_ else 'Other'])[0] if le['brand'] else 0

    row = {
        'model_year': model_year, 'milage': milage, 'car_age': car_age,
        'age_times_milage': age_times_milage, 'fuel_type_enc': fuel_enc,
        'transmission_enc': trans_enc, 'brand_enc': brand_enc,
        'has_accident': has_accident, 'clean_title_flag': clean_title,
        'engine_hp': engine_hp, 'condition_score': condition_score
    }
    X_single = pd.DataFrame([{f: row.get(f, 0) for f in features}])
    return round(model.predict(X_single)[0], 2)


# Example: 2019 BMW, 45k miles, Gasoline, Automatic, no accident, condition = minor damage (1)
price = predict_car_price(
    model_payload,
    model_year=2019, milage=45000, fuel_type='Gasoline', transmission='Automatic',
    brand='BMW', has_accident=0, clean_title=1, engine_hp=255, condition_score=1
)
print(f'Predicted price: ${price:,.0f}')